# NB29 — KANSER Panel Refinement: 6 Stratejiler Karşılaştırması

KANSER panelinin %80/20 F1 skorunu 0.716'dan (NB16 stack_lr) yükseltmek için
NB21 (PAH refinement) protokolünü KANSER'e uyarlıyoruz.

**Avantaj:** KANSER n=120 benign (CFTR'nin 5×), %69 pathogenic (en dengeli panel) → istatistiksel olarak anlamlı sonuçlar.

**6 Deney:**
| # | Deney | Açıklama |
|---|-------|----------|
| E0 | Baseline | KANSER-only LightGBM (yeni protokol: AUPRC + LOO-MCC) |
| E1 | COMBINED | MASTER+PAH+CFTR ile eğit, KANSER üzerinde LOO-CV |
| E2 | BalancedBagging | Sweep: n_estimators ∈ {10,20,30,50} × max_features ∈ {0.7,0.85,1.0} |
| E3 | Calibrate-then-shift | Raw + Platt + BalBag, her biri + Saerens prior-shift |
| E4 | Heterogeneous stacking | LGBM+XGB+CB+RF → OOF → LR meta |
| E5 | Missing-aware dual | Model M3+ (flag dahil) + M3− (AL_16..AL_25 hariç), ensemble |

**Referans:** NB16 KANSER stack_lr → Boot %80/20 F1=0.716, CI=[0.67–0.77]

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# imblearn yüklü mü kontrol et
try:
    from imblearn.ensemble import BalancedBaggingClassifier
    HAS_IMBLEARN = True
except ImportError:
    print("[UYARI] imblearn bulunamadı, BalancedBagging deney atlanacak.")
    HAS_IMBLEARN = False

# catboost yüklü mü kontrol et
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    print("[UYARI] catboost bulunamadı, E4 heterogeneous stacking sınırlı olacak.")
    HAS_CATBOOST = False

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR
from src.metrics import compute_all_metrics, optimize_threshold

from sklearn.model_selection import LeaveOneOut, StratifiedKFold, StratifiedShuffleSplit, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

np.random.seed(SEED)

# Sabitler
PI_TEST = 0.20           # final test: %20 pathogenic
FINAL_BENIGN_FRAC = 0.80 # final test: %80 benign
N_BOOT = 50              # bootstrap tekrar sayısı
BOOT_SEED = 123
N_MULTISEED = 10         # multi-seed 50/50 değerlendirme

# Target panel
TARGET_PANEL = "KANSER"

# Sonuç dizini
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v14_kanser_refinement")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR_NB, exist_ok=True)

print(f"NB29 -- KANSER Panel Refinement")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")
print(f"HAS_IMBLEARN={HAS_IMBLEARN}, HAS_CATBOOST={HAS_CATBOOST}")

NB29 -- KANSER Panel Refinement
SEED=42, PI_TEST=0.2, N_BOOT=50
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v14_kanser_refinement
HAS_IMBLEARN=True, HAS_CATBOOST=True


In [2]:
# Cell 2: Veri Yükleme + Sütun Temizliği (NB21 ile aynı mantık)
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

# --- Veri yükleme ---
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr   = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah    = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

# KANSER etiket dağılımını doğrula (to-do.md'de "pos=268, neg=120" yazıyor)
kanser_pos = (df_kanser[TARGET] == 1).sum()
kanser_neg = (df_kanser[TARGET] == 0).sum()
print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={kanser_pos}, neg={kanser_neg}) ← Doğrula!")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# --- COMBINED eğitim havuzu: MASTER + PAH + CFTR (KANSER HARİÇ!) ---
df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+PAH+CFTR): {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})")

# --- Cross-panel birebir-aynı satır drop ---
feat_cols = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Tüm feature+label birebir aynı olan satırların panel ID'lerini döndür."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_kanser, df_master, feat_cols, TARGET)
if dup_ids:
    df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"KANSER: {len(dup_ids)} birebir-aynı satır drop edildi -> {df_kanser.shape}")
else:
    print("KANSER: birebir-aynı satır yok")

# --- Sütun temizliği ---
# Sabit sütunlar (MASTER üzerinde)
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

# Özdeş çift sütunlar (MASTER üzerinde)
def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}")

# Her dataset'ten drop
keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()

df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, KANSER={df_kanser.shape}")

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120) ← Doğrula!
CFTR:   (111, 353)   (pos=90, neg=21)
PAH:    (372, 353)  (pos=310, neg=62)

COMBINED (MASTER+PAH+CFTR): (3414, 353) (pos=2549, neg=865)
KANSER: 3 birebir-aynı satır drop edildi -> (385, 353)
Constant: 0, Duplicate pairs: 58 -> drop 58
Toplam drop: 58, Kalan feature: 293

Final shapes: MASTER=(2931, 295), COMBINED=(3414, 295), KANSER=(385, 295)


In [3]:
# Cell 3: FE + M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    """Train üzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing tespit
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    # Median (train üzerinde)
    medians = X[num_cols].median()
    
    # Kategorik fill + LE
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # is_missing flag (high miss sütunlar için)
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    
    # Median imputation
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    
    # Kategorik
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        # Unseen kategoriler -> -1
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

# Preprocessor fit (MASTER üzerinde -- tüm stratejiler için ortak)
prep_master = fit_preprocessor(df_master, keep_cols, TARGET)
prep_combined = fit_preprocessor(df_combined, keep_cols, TARGET)

# Transform
X_master_df = transform_X(df_master, keep_cols, prep_master)
y_master = df_master[TARGET].values

X_combined_df = transform_X(df_combined, keep_cols, prep_combined)
y_combined = df_combined[TARGET].values

X_kanser_master_df = transform_X(df_kanser, keep_cols, prep_master)
X_kanser_combined_df = transform_X(df_kanser, keep_cols, prep_combined)
y_kanser = df_kanser[TARGET].values

print(f"X_master: {X_master_df.shape}, X_combined: {X_combined_df.shape}")
print(f"X_kanser (master prep): {X_kanser_master_df.shape}, X_kanser (combined prep): {X_kanser_combined_df.shape}")
print(f"KANSER label dist: pos={y_kanser.sum()}, neg={(y_kanser==0).sum()}")

X_master: (2931, 434), X_combined: (3414, 434)
X_kanser (master prep): (385, 434), X_kanser (combined prep): (385, 434)
KANSER label dist: pos=265, neg=120


In [4]:
# Cell 4: Değerlendirme Altyapısı -- LOO-CV, bootstrap %80/20, prior shift, AUPRC

# --- 4a. Prior shift düzeltmesi (Saerens et al. 2002) ---
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# --- 4b. Bootstrap %80/20 ---
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020(y, prob):
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best:
            best, best_thr = f, thr
    return float(best_thr)

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    """N resample ortalamasıyla robust threshold seç."""
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

# --- 4c. LOO-CV metrikleri (AUPRC dahil) ---
def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1  = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    auprc = average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "auprc": auprc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "y_pred": y_pred, "y_true": np.asarray(y_true), "prob": prob}

# --- 4d. Multi-seed 50/50 değerlendirme ---
def multiseed_eval(fit_predict_fn, X_panel, y_panel, n_seeds=N_MULTISEED):
    """fit_predict_fn(X_tr, y_tr, X_te, y_te) -> test_proba"""
    f1s = []
    for seed_i in range(n_seeds):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED + seed_i)
        for tri, tei in sss.split(X_panel, y_panel):
            X_tr = X_panel.iloc[tri]
            y_tr = y_panel[tri]
            X_te = X_panel.iloc[tei]
            y_te = y_panel[tei]
            p_te = fit_predict_fn(X_tr, y_tr, X_te, y_te)
            thr = select_threshold_8020(y_te, p_te)
            f1s.append(_f1_pos(y_te, (p_te >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()), "scores": f1s.tolist()}

# --- 4e. Train metrikleri ---
def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print("Değerlendirme altyapısı hazır.")

Değerlendirme altyapısı hazır.


In [5]:
# Cell 5: Model Yardımcıları

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbosity": 0,
    "n_jobs": -1
}

def _lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def _xgb_classifier(**kw):
    params = {**XGB_PARAMS, **kw}
    return XGBClassifier(**params)

def _rf_classifier(**kw):
    params = {"n_estimators": 200, "max_depth": 15, "random_state": SEED, "n_jobs": -1}
    params.update(kw)
    return RandomForestClassifier(**params)

def _le_encode_for_loo(X_df):
    """Kategorik sütunları basit label-encode et (LOO uyumlu)."""
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

# HATA 1 DÜZELTME: OOF yardımcı fonksiyonu
def oof_predict(make_model_fn, X_df, y, n_splits=5, seed=SEED):
    """Gerçek out-of-fold tahmin: her örnek, onu görmemiş fold modelinden tahmin alır.
    make_model_fn() çağrıldığında YENİ (fit edilmemiş) bir model döndürmeli.
    Döndürür: (oof_proba [test için], full_model [train metrikleri için])."""
    oof = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tri, vai in skf.split(X_df, y):
        m = make_model_fn()
        m.fit(X_df.iloc[tri], y[tri])
        oof[vai] = m.predict_proba(X_df.iloc[vai])[:, 1]
    full = make_model_fn()
    full.fit(X_df, y)
    return oof, full

print("Model yardımcıları hazır.")

Model yardımcıları hazır.


In [6]:
# Cell 6-11: 6 Deney (Exp 0-5)
print("="*70)
print("NB29 -- KANSER Refinement: 6 Deney Karşılaştırması")
print("="*70)

all_results = {}
sweep_results = []  # Exp2 için sweep sonuçları

# Pre-encode
X_kanser_m_le, _ = _le_encode_for_loo(X_kanser_master_df)
X_kanser_c_le, _ = _le_encode_for_loo(X_kanser_combined_df)
X_master_le, _ = _le_encode_for_loo(X_master_df)
X_combined_le, _ = _le_encode_for_loo(X_combined_df)

# ================================================================
# E0: Baseline -- KANSER-only LightGBM (GERÇEK OOF)
# ================================================================
print("\n[E0] Baseline -- KANSER-only LightGBM (OOF)...")
pi_kanser = float(y_kanser.mean())

# HATA 1 DÜZELTME: E0 OOF kullan
oof_e0, full_e0 = oof_predict(lambda: _lgbm_classifier(), X_kanser_m_le, y_kanser)
p_e0_train = full_e0.predict_proba(X_kanser_m_le)[:, 1]  # Train metrikleri için
p_e0_kanser = oof_e0  # TEST değerlendirmesi için OOF

thr_e0 = select_threshold_8020_robust(y_kanser, p_e0_kanser)
train_e0 = train_metrics_at(y_kanser, p_e0_train, thr_e0)
loo_e0_raw = loo_metrics(y_kanser, p_e0_kanser, prior_shift=False)
loo_e0_prior = loo_metrics(y_kanser, p_e0_kanser, prior_shift=True, pi_train=pi_kanser)

all_results["E0_Baseline"] = {
    "loo_raw": loo_e0_raw, "loo_prior": loo_e0_prior,
    "train": train_e0, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  MCC(raw)={loo_e0_raw['mcc']:.4f} MCC(prior)={loo_e0_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e0_prior['boot8020']['mean']:.4f} +/- {loo_e0_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e0_prior['auprc']:.4f}")

# ================================================================
# E1: COMBINED -- MASTER+PAH+CFTR ile eğit, KANSER test
# ================================================================
print("\n[E1] COMBINED (MASTER+PAH+CFTR)...")
pi_combined = float(y_combined.mean())
m_e1 = _lgbm_classifier()
m_e1.fit(X_combined_le, y_combined)
p_e1_train = m_e1.predict_proba(X_combined_le)[:, 1]
p_e1_kanser = m_e1.predict_proba(X_kanser_c_le)[:, 1]

thr_e1 = select_threshold_8020_robust(y_combined, p_e1_train)
train_e1 = train_metrics_at(y_combined, p_e1_train, thr_e1)
loo_e1_raw = loo_metrics(y_kanser, p_e1_kanser, prior_shift=False)
loo_e1_prior = loo_metrics(y_kanser, p_e1_kanser, prior_shift=True, pi_train=pi_combined)

all_results["E1_COMBINED"] = {
    "loo_raw": loo_e1_raw, "loo_prior": loo_e1_prior,
    "train": train_e1, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_e1_raw['mcc']:.4f} MCC(prior)={loo_e1_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_e1_prior['boot8020']['mean']:.4f} +/- {loo_e1_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_e1_prior['auprc']:.4f}")

NB29 -- KANSER Refinement: 6 Deney Karşılaştırması

[E0] Baseline -- KANSER-only LightGBM (OOF)...
  MCC(raw)=0.6571 MCC(prior)=0.5683
  Boot-mean(prior)=0.6621 +/- 0.0577
  AUPRC=0.9290

[E1] COMBINED (MASTER+PAH+CFTR)...
  MCC(raw)=0.6496 MCC(prior)=0.6542
  Boot-mean(prior)=0.7143 +/- 0.0412
  AUPRC=0.9643


In [7]:
# Cell 7: E2 -- BalancedBagging Sweep (GERÇEK OOF)
# Defansif init: imblearn yoksa ya da hücre sırasız çalışırsa E3-C NameError vermesin
best_config = None
best_n_est = None
best_max_feat = None

if HAS_IMBLEARN:
    print("\n[E2] BalancedBagging Sweep (OOF)...")
    sweep_results = []
    best_mcc = -2.0
    
    for n_est in [10, 20, 30, 50]:
        for max_feat in [0.7, 0.85, 1.0]:
            config_name = f"n_est={n_est}_maxfeat={max_feat}"
            
            # HATA 1 DÜZELTME: E2 OOF kullan
            def make_bb_model(n_e=n_est, mf=max_feat):
                return BalancedBaggingClassifier(
                    estimator=_lgbm_classifier(),
                    n_estimators=n_e,
                    sampling_strategy="not minority",
                    max_features=mf,
                    random_state=SEED,
                    n_jobs=-1
                )
            
            oof_e2, full_e2 = oof_predict(make_bb_model, X_kanser_m_le, y_kanser)
            p_e2_train = full_e2.predict_proba(X_kanser_m_le)[:, 1]  # Train metrikleri için
            p_e2_test = oof_e2  # TEST değerlendirmesi için OOF
            
            thr = select_threshold_8020_robust(y_kanser, p_e2_test)
            train_m = train_metrics_at(y_kanser, p_e2_train, thr)
            loo_raw = loo_metrics(y_kanser, p_e2_test, prior_shift=False)
            loo_prior = loo_metrics(y_kanser, p_e2_test, prior_shift=True, pi_train=pi_kanser)
            
            sweep_results.append({
                "config": config_name,
                "n_est": n_est,
                "max_feat": max_feat,
                "mcc_raw": loo_raw["mcc"],
                "mcc_prior": loo_prior["mcc"],
                "boot_mean": loo_prior["boot8020"]["mean"],
                "boot_std": loo_prior["boot8020"]["std"]
            })
            
            if loo_prior["mcc"] > best_mcc:
                best_mcc = loo_prior["mcc"]
                best_config = config_name
                best_n_est = n_est
                best_max_feat = max_feat
                best_res = {"mcc": loo_prior["mcc"], "boot_mean": loo_prior["boot8020"]["mean"],
                           "train": train_m, "loo_raw": loo_raw, "loo_prior": loo_prior}
    
    all_results["E2_BalancedBagging"] = {
        "loo_raw": best_res["loo_raw"], "loo_prior": best_res["loo_prior"],
        "train": best_res["train"], "pi_train": pi_kanser, "n_train": len(y_kanser)
    }
    print(f"  En iyi config: {best_config}")
    print(f"  MCC(prior)={best_mcc:.4f}, Boot-mean={best_res['boot_mean']:.4f}")
    print(f"  AUPRC={best_res['loo_prior']['auprc']:.4f}")
else:
    print("\n[E2] BalancedBagging atlanıyor (imblearn yüklü değil)")
    all_results["E2_BalancedBagging"] = None


[E2] BalancedBagging Sweep (OOF)...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  En iyi config: n_est=10_maxfeat=1.0
  MCC(prior)=0.6378, Boot-mean=0.7070
  AUPRC=0.9398


In [8]:
# Cell 8: E3 -- Calibrate-then-shift (3 strateji, GERÇEK OOF)
print("\n[E3] Calibrate-then-shift (Alexandari 2020)...")

# A) Raw + Prior-shift (OOF)
def make_lgbm_model():
    return _lgbm_classifier()

oof_ref, full_ref = oof_predict(make_lgbm_model, X_kanser_m_le, y_kanser)
p_ref_train = full_ref.predict_proba(X_kanser_m_le)[:, 1]  # Train metrikleri için
p_ref_test = oof_ref  # TEST için OOF

thr_ref = select_threshold_8020_robust(y_kanser, p_ref_test)
train_ref = train_metrics_at(y_kanser, p_ref_train, thr_ref)
loo_ref_raw = loo_metrics(y_kanser, p_ref_test, prior_shift=False)
loo_ref_prior = loo_metrics(y_kanser, p_ref_test, prior_shift=True, pi_train=pi_kanser)

all_results["E3_A_Raw_PriorShift"] = {
    "loo_raw": loo_ref_raw, "loo_prior": loo_ref_prior,
    "train": train_ref, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  A) Raw + Prior-shift: MCC={loo_ref_prior['mcc']:.4f}, Boot={loo_ref_prior['boot8020']['mean']:.4f}")

# B) Platt kalibrasyon + prior-shift (OOF)
def make_platt_model():
    return CalibratedClassifierCV(estimator=_lgbm_classifier(), method="sigmoid", cv=5)

oof_platt, full_platt = oof_predict(make_platt_model, X_kanser_m_le, y_kanser)
p_platt_train = full_platt.predict_proba(X_kanser_m_le)[:, 1]  # Train metrikleri için
p_platt_test = oof_platt  # TEST için OOF

thr_platt = select_threshold_8020_robust(y_kanser, p_platt_test)
train_platt = train_metrics_at(y_kanser, p_platt_train, thr_platt)
loo_platt_raw = loo_metrics(y_kanser, p_platt_test, prior_shift=False)
loo_platt_prior = loo_metrics(y_kanser, p_platt_test, prior_shift=True, pi_train=pi_kanser)

all_results["E3_B_Platt_PriorShift"] = {
    "loo_raw": loo_platt_raw, "loo_prior": loo_platt_prior,
    "train": train_platt, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  B) Platt + Prior-shift: MCC={loo_platt_prior['mcc']:.4f}, Boot={loo_platt_prior['boot8020']['mean']:.4f}")

# C) BalancedBagging + Platt + Prior-shift (best E2 config ile, OOF)
# HATA 2 DÜZELTME: best_n_est ve best_max_feat int/float olarak kullan, string parse etme
if HAS_IMBLEARN and best_config:
    def make_bbc_model():
        return CalibratedClassifierCV(
            estimator=BalancedBaggingClassifier(
                estimator=_lgbm_classifier(),
                n_estimators=best_n_est,  # int olarak
                sampling_strategy="not minority",
                max_features=best_max_feat,  # float olarak
                random_state=SEED,
                n_jobs=-1
            ),
            method="sigmoid",
            cv=5
        )
    
    oof_bbc, full_bbc = oof_predict(make_bbc_model, X_kanser_m_le, y_kanser)
    p_bbc_train = full_bbc.predict_proba(X_kanser_m_le)[:, 1]  # Train metrikleri için
    p_bbc_test = oof_bbc  # TEST için OOF
    
    thr_bbc = select_threshold_8020_robust(y_kanser, p_bbc_test)
    train_bbc = train_metrics_at(y_kanser, p_bbc_train, thr_bbc)
    loo_bbc_raw = loo_metrics(y_kanser, p_bbc_test, prior_shift=False)
    loo_bbc_prior = loo_metrics(y_kanser, p_bbc_test, prior_shift=True, pi_train=pi_kanser)
    
    all_results["E3_C_BalBag_Platt_PriorShift"] = {
        "loo_raw": loo_bbc_raw, "loo_prior": loo_bbc_prior,
        "train": train_bbc, "pi_train": pi_kanser, "n_train": len(y_kanser)
    }
    print(f"  C) BalBag + Platt + Prior-shift (n_est={best_n_est}, max_feat={best_max_feat}): MCC={loo_bbc_prior['mcc']:.4f}, Boot={loo_bbc_prior['boot8020']['mean']:.4f}")
else:
    print(f"  C) BalBag + Platt atlanıyor")


[E3] Calibrate-then-shift (Alexandari 2020)...
  A) Raw + Prior-shift: MCC=0.5683, Boot=0.6621
  B) Platt + Prior-shift: MCC=0.6273, Boot=0.6845


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  C) BalBag + Platt + Prior-shift (n_est=10, max_feat=1.0): MCC=0.6297, Boot=0.7067


In [9]:
# Cell 9: E4 -- Heterogeneous Stacking (LGBM+XGB+CB+RF → LR meta, GERÇEK OOF)
print("\n[E4] Heterogeneous Stacking (LGBM+XGB+CB+RF)...")

# HATA 1 DÜZELTME: Base OOF tahmin + Meta OOF
skf_stack = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
n_bases = 4 if HAS_CATBOOST else 3
oof_preds = np.zeros((len(y_kanser), n_bases))

base_models = [_lgbm_classifier(), _xgb_classifier(), _rf_classifier()]
if HAS_CATBOOST:
    base_models.append(CatBoostClassifier(iterations=300, verbose=0, random_state=SEED))

# Base model OOF tahmin
for bi, model in enumerate(base_models):
    for tri, vai in skf_stack.split(X_kanser_m_le, y_kanser):
        m = deepcopy(model)
        X_tr, y_tr = X_kanser_m_le.iloc[tri], y_kanser[tri]
        X_va = X_kanser_m_le.iloc[vai]
        m.fit(X_tr, y_tr)
        oof_preds[vai, bi] = m.predict_proba(X_va)[:, 1]

# Meta: Logistic Regression OOF (meta seviyesinde de leakage var)
def make_meta_lr():
    return LogisticRegression(C=1.0, penalty="l2", class_weight="balanced", random_state=SEED, max_iter=1000)

oof_meta, full_meta = oof_predict(make_meta_lr, pd.DataFrame(oof_preds), y_kanser)
p_meta_train = full_meta.predict_proba(pd.DataFrame(oof_preds))[:, 1]  # Train metrikleri için
p_meta_test = oof_meta  # TEST için OOF

thr_meta = select_threshold_8020_robust(y_kanser, p_meta_test)
train_meta = train_metrics_at(y_kanser, p_meta_train, thr_meta)
loo_meta_raw = loo_metrics(y_kanser, p_meta_test, prior_shift=False)
loo_meta_prior = loo_metrics(y_kanser, p_meta_test, prior_shift=True, pi_train=pi_kanser)

all_results["E4_Heterogeneous_Stack"] = {
    "loo_raw": loo_meta_raw, "loo_prior": loo_meta_prior,
    "train": train_meta, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  MCC(raw)={loo_meta_raw['mcc']:.4f} MCC(prior)={loo_meta_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_meta_prior['boot8020']['mean']:.4f} +/- {loo_meta_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_meta_prior['auprc']:.4f}")


[E4] Heterogeneous Stacking (LGBM+XGB+CB+RF)...
  MCC(raw)=0.5401 MCC(prior)=0.6139
  Boot-mean(prior)=0.6667 +/- 0.0379
  AUPRC=0.9304


In [10]:
# Cell 10: E5 -- Missing-Aware Dual Model (GERÇEK OOF)
print("\n[E5] Missing-Aware Dual Model (M3+ ve M3−)...")

# M3+ : is_missing flag dahil
# M3− : AL_16..AL_25 + is_missing flag'leri hariç

def get_missingness_risky_cols(keep_cols):
    return [c for c in CR.AL_MISSINGNESS_LEAKAGE_RISK if c in keep_cols]

risk_cols = get_missingness_risky_cols(keep_cols)
missing_flag_cols = [c for c in X_kanser_m_le.columns if c.startswith("is_missing_")]

# M3+: Tüm sütunlar (original, OOF)
X_k_m3p = X_kanser_m_le.copy()

def make_m3p_model():
    return _lgbm_classifier()

oof_m3p, full_m3p = oof_predict(make_m3p_model, X_k_m3p, y_kanser)
p_m3p_train = full_m3p.predict_proba(X_k_m3p)[:, 1]
p_m3p_test = oof_m3p

# M3−: AL_MISSINGNESS_LEAKAGE_RISK + is_missing_* hariç (OOF)
drop_cols_m3m = risk_cols + missing_flag_cols
X_k_m3m = X_kanser_m_le.drop(columns=[c for c in drop_cols_m3m if c in X_kanser_m_le.columns])

def make_m3m_model():
    return _lgbm_classifier()

oof_m3m, full_m3m = oof_predict(make_m3m_model, X_k_m3m, y_kanser)
p_m3m_train = full_m3m.predict_proba(X_k_m3m)[:, 1]
p_m3m_test = oof_m3m

# Ensemble: ortalama
p_ens_train = (p_m3p_train + p_m3m_train) / 2.0
p_ens_test = (p_m3p_test + p_m3m_test) / 2.0

thr_ens = select_threshold_8020_robust(y_kanser, p_ens_test)
train_ens = train_metrics_at(y_kanser, p_ens_train, thr_ens)
loo_ens_raw = loo_metrics(y_kanser, p_ens_test, prior_shift=False)
loo_ens_prior = loo_metrics(y_kanser, p_ens_test, prior_shift=True, pi_train=pi_kanser)

all_results["E5_Missing_Dual"] = {
    "loo_raw": loo_ens_raw, "loo_prior": loo_ens_prior,
    "train": train_ens, "pi_train": pi_kanser, "n_train": len(y_kanser)
}
print(f"  MCC(raw)={loo_ens_raw['mcc']:.4f} MCC(prior)={loo_ens_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_ens_prior['boot8020']['mean']:.4f} +/- {loo_ens_prior['boot8020']['std']:.4f}")
print(f"  AUPRC={loo_ens_prior['auprc']:.4f}")
print(f"  Drop sütun (risk+flags): {len(drop_cols_m3m)}, Kalan: {X_k_m3m.shape[1]}")

print("\n" + "="*70)
print("Tüm deneyler tamamlandı!")
print("="*70)


[E5] Missing-Aware Dual Model (M3+ ve M3−)...
  MCC(raw)=0.6594 MCC(prior)=0.6111
  Boot-mean(prior)=0.6789 +/- 0.0446
  AUPRC=0.9292
  Drop sütun (risk+flags): 151, Kalan: 283

Tüm deneyler tamamlandı!


In [11]:
# Cell 11: Sonuç Derleme

rows = []
for name, res in all_results.items():
    if res is None:
        continue
    loo_p = res["loo_prior"]
    loo_r = res["loo_raw"]
    train = res["train"]
    boot = loo_p["boot8020"]
    rows.append({
        "Deney": name,
        "n_train": res["n_train"],
        "LOO-MCC (raw)": round(loo_r["mcc"], 4),
        "LOO-MCC (prior)": round(loo_p["mcc"], 4),
        "LOO-F1": round(loo_p["f1"], 4),
        "LOO-AUC": round(loo_p["auc"], 4),
        "AUPRC": round(loo_p["auprc"], 4),
        "Precision": round(loo_p["precision"], 4),
        "Recall": round(loo_p["recall"], 4),
        "Boot-mean": round(boot["mean"], 4),
        "Boot-std": round(boot["std"], 4),
        "Boot-lo": round(boot["lo"], 4),
        "Boot-hi": round(boot["hi"], 4),
        "Train-F1": round(train["train_f1"], 4),
        "Train-MCC": round(train["train_mcc"], 4),
        "TN": loo_p["tn"], "FP": loo_p["fp"],
        "FN": loo_p["fn"], "TP": loo_p["tp"],
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values("LOO-MCC (prior)", ascending=False).reset_index(drop=True)

print("\n=== KANSER Refinement Sonuçları (LOO-MCC prior sıralama) ===\n")
display_cols = ["Deney", "n_train", "LOO-MCC (prior)", "Boot-mean", "Boot-std", "AUPRC", "Precision", "Recall", "TN", "FP", "FN", "TP"]
print(results_df[display_cols].to_string(index=False))

# CSV kaydet
csv_path = os.path.join(RESULTS_DIR, "kanser_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\nSonuçlar kaydedildi: {csv_path}")

# Sweep sonuçları (Exp2) — HATA 4 DÜZELTME: sweep_results kontrol
if sweep_results:
    sweep_df = pd.DataFrame(sweep_results)
    sweep_csv = os.path.join(RESULTS_DIR, "sweep_results.csv")
    sweep_df.to_csv(sweep_csv, index=False)
    print(f"Sweep sonuçları: {sweep_csv}")
else:
    print("Sweep sonuçları yok (imblearn yüklü değil)")

print("\n--- Referans: NB16 KANSER stack_lr Boot %80/20 F1=0.716, CI=[0.67-0.77] ---")


=== KANSER Refinement Sonuçları (LOO-MCC prior sıralama) ===

                       Deney  n_train  LOO-MCC (prior)  Boot-mean  Boot-std  AUPRC  Precision  Recall  TN  FP  FN  TP
                 E1_COMBINED     3414           0.6542     0.7143    0.0412 0.9643     0.9425  0.8038 107  13  52 213
          E2_BalancedBagging      385           0.6378     0.7070    0.0353 0.9398     0.9414  0.7887 107  13  56 209
E3_C_BalBag_Platt_PriorShift      385           0.6297     0.7067    0.0407 0.9357     0.9409  0.7811 107  13  58 207
       E3_B_Platt_PriorShift      385           0.6273     0.6845    0.0402 0.9195     0.9333  0.7925 105  15  55 210
      E4_Heterogeneous_Stack      385           0.6139     0.6667    0.0379 0.9304     0.9217  0.8000 102  18  53 212
             E5_Missing_Dual      385           0.6111     0.6789    0.0446 0.9292     0.9321  0.7774 105  15  59 206
                 E0_Baseline      385           0.5683     0.6621    0.0577 0.9290     0.9363  0.7208 107  13  

In [12]:
# Cell 12: Görsellendirmeler

valid_res = {k: v for k, v in all_results.items() if v is not None}

# --- Fig 1: LOO-MCC karşılaştırması (raw vs prior-shift) ---
fig, ax = plt.subplots(figsize=(12, 5))
names = list(valid_res.keys())
mcc_raw = [valid_res[n]["loo_raw"]["mcc"] for n in names]
mcc_prior = [valid_res[n]["loo_prior"]["mcc"] for n in names]
x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, mcc_raw, w, label="MCC (raw)", color="steelblue", alpha=0.8)
bars2 = ax.bar(x + w/2, mcc_prior, w, label="MCC (prior-shift)", color="darkorange", alpha=0.8)
ax.set_ylabel("LOO-CV MCC")
ax.set_title("NB29 -- KANSER: LOO-CV MCC Karşılaştırması")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.legend()
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
for b in bars1:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
for b in bars2:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_loo_mcc.png"), dpi=150)
plt.close()
print("fig1_loo_mcc.png kaydedildi")

# --- Fig 2: Boot-mean karşılaştırması ---
fig, ax = plt.subplots(figsize=(12, 5))
boot_means = [valid_res[n]["loo_prior"]["boot8020"]["mean"] for n in names]
boot_stds = [valid_res[n]["loo_prior"]["boot8020"]["std"] for n in names]
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
bars = ax.bar(x, boot_means, yerr=boot_stds, capsize=5, color=colors, alpha=0.85)
ax.set_ylabel("Bootstrap %80/20 F1 (pathogenic)")
ax.set_title("NB29 -- KANSER: Bootstrap %80/20 F1 (N=50, prior-shift)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.axhline(y=0.716, color="red", linestyle="--", alpha=0.5, label="NB16 baseline (0.716)")
ax.legend()
for b in bars:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_boot_mean.png"), dpi=150)
plt.close()
print("fig2_boot_mean.png kaydedildi")

# --- Fig 3: AUPRC karşılaştırması ---
fig, ax = plt.subplots(figsize=(12, 5))
auprc_vals = [valid_res[n]["loo_prior"]["auprc"] for n in names]
bars = ax.bar(x, auprc_vals, color=colors, alpha=0.85)
ax.set_ylabel("AUPRC (Literature Primary Metric)")
ax.set_title("NB29 -- KANSER: AUPRC Karşılaştırması")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.set_ylim([0, 1])
for b in bars:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_auprc.png"), dpi=150)
plt.close()
print("fig3_auprc.png kaydedildi")

# --- Fig 4: Confusion matrix (en iyi 3 strateji) ---
sorted_names = results_df["Deney"].tolist()[:3]
fig, axes = plt.subplots(1, min(3, len(sorted_names)), figsize=(5*min(3, len(sorted_names)), 4))
if len(sorted_names) == 1:
    axes = [axes]
for i, name in enumerate(sorted_names):
    res = all_results[name]["loo_prior"]
    cm = np.array([[res["tn"], res["fp"]], [res["fn"], res["tp"]]])
    ax = axes[i]
    im = ax.imshow(cm, cmap="Blues", aspect="auto")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Patho"])
    ax.set_yticklabels(["Benign", "Patho"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    boot = all_results[name]["loo_prior"]["boot8020"]
    ax.set_title(f"{name}\nMCC={res['mcc']:.3f}, Boot={boot['mean']:.3f}", fontsize=9)
    for ii in range(2):
        for jj in range(2):
            ax.text(jj, ii, str(cm[ii, jj]), ha="center", va="center", fontsize=14,
                   color="white" if cm[ii, jj] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_confusion.png"), dpi=150)
plt.close()
print("fig4_confusion.png kaydedildi")

fig1_loo_mcc.png kaydedildi
fig2_boot_mean.png kaydedildi
fig3_auprc.png kaydedildi
fig4_confusion.png kaydedildi


In [13]:
# Cell 13: Özet & Tartışma

print("\n" + "="*80)
print("NB29 ÖZET & TARTIŞMA")
print("="*80)

# En iyi strateji
best = results_df.iloc[0]
print(f"\nEn iyi strateji: {best['Deney']}")
print(f"  LOO-MCC (prior): {best['LOO-MCC (prior)']}")
print(f"  Boot-mean @%80/20: {best['Boot-mean']} +/- {best['Boot-std']}")
print(f"  Boot CI: [{best['Boot-lo']}, {best['Boot-hi']}]")
print(f"  AUPRC (mutlak, literatür birincil metriği): {best['AUPRC']:.4f}")
print(f"  Precision: {best['Precision']}, Recall: {best['Recall']}")
print(f"  Confusion: TN={best['TN']}, FP={best['FP']}, FN={best['FN']}, TP={best['TP']}")

print(f"\nNB16 referans: KANSER stack_lr Boot %80/20 F1=0.716")
delta = best['Boot-mean'] - 0.716
print(f"Fark: {delta:+.4f} (NB29 - NB16)")

# HATA 3 DÜZELTME: Uydurma 0.850 baseline'dan kurtul
print(f"\n[NOT] NB16'nın AUPRC'si ölçülmedi (sadece Boot F1 raporlandı).")
print(f"Bu nedenle delta AUPRC hesaplanamaz. Sonuçları mutlak değer olarak raporla.")

print("\n--- Tüm deneyler (prior-shift MCC sıralama) ---")
for _, row in results_df.iterrows():
    print(f"  {row['Deney']:35s} MCC={row['LOO-MCC (prior)']:.4f}  Boot={row['Boot-mean']:.4f}+/-{row['Boot-std']:.4f}  AUPRC={row['AUPRC']:.4f}  P={row['Precision']:.3f} R={row['Recall']:.3f}")

# Prior-shift etkisi
print("\n--- Prior-shift etkisi ---")
for name in all_results:
    if all_results[name] is None:
        continue
    raw = all_results[name]["loo_raw"]["mcc"]
    prior = all_results[name]["loo_prior"]["mcc"]
    delta_ps = prior - raw
    print(f"  {name:35s} raw={raw:.4f} -> prior={prior:.4f}  delta={delta_ps:+.4f}")

# Overfit kontrolü
print("\n--- Overfit kontrolü (train F1 vs LOO F1) ---")
for name in all_results:
    if all_results[name] is None:
        continue
    tf = all_results[name]["train"]["train_f1"]
    lf = all_results[name]["loo_prior"]["f1"]
    gap = tf - lf
    status = "OK" if gap < 0.15 else "ORTA" if gap < 0.25 else "YUKSEK"
    print(f"  {name:35s} train={tf:.4f} loo={lf:.4f} gap={gap:.4f} [{status}]")

print("\nDok: KANSER %69 pathogenic, n=120 benign (en dengeli panel) → CI dar, istatistiksel olarak güvenilir.")


NB29 ÖZET & TARTIŞMA

En iyi strateji: E1_COMBINED
  LOO-MCC (prior): 0.6542
  Boot-mean @%80/20: 0.7143 +/- 0.0412
  Boot CI: [0.6562, 0.7887]
  AUPRC (mutlak, literatür birincil metriği): 0.9643
  Precision: 0.9425, Recall: 0.8038
  Confusion: TN=107, FP=13, FN=52, TP=213

NB16 referans: KANSER stack_lr Boot %80/20 F1=0.716
Fark: -0.0017 (NB29 - NB16)

[NOT] NB16'nın AUPRC'si ölçülmedi (sadece Boot F1 raporlandı).
Bu nedenle delta AUPRC hesaplanamaz. Sonuçları mutlak değer olarak raporla.

--- Tüm deneyler (prior-shift MCC sıralama) ---
  E1_COMBINED                         MCC=0.6542  Boot=0.7143+/-0.0412  AUPRC=0.9643  P=0.943 R=0.804
  E2_BalancedBagging                  MCC=0.6378  Boot=0.7070+/-0.0353  AUPRC=0.9398  P=0.941 R=0.789
  E3_C_BalBag_Platt_PriorShift        MCC=0.6297  Boot=0.7067+/-0.0407  AUPRC=0.9357  P=0.941 R=0.781
  E3_B_Platt_PriorShift               MCC=0.6273  Boot=0.6845+/-0.0402  AUPRC=0.9195  P=0.933 R=0.792
  E4_Heterogeneous_Stack              MCC=0.61

In [14]:
# Cell 14: PDF Rapor (fpdf2)
from fpdf import FPDF
from datetime import datetime

class KanserRefinementReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "KANSER (Hereditary Cancer) Paneli Refinement Raporu | TEKNOFEST 2025", 0, 1, "C")
        self.set_draw_color(0, 102, 153)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", 0, 0, "C")

    def section(self, title):
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)

    def body(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)
        self.ln(2)

    def bullet(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.cell(5, 5, "-")
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)

    def add_table(self, headers, data, col_widths=None):
        if col_widths is None:
            col_widths = [190 / len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        self.set_fill_color(0, 102, 153)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, h, 1, 0, "C", True)
        self.ln()
        self.set_x(self.l_margin)
        self.set_text_color(0, 0, 0)
        self.set_font("Helvetica", "", 7)
        for row in data:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, "C")
            self.ln()
            self.set_x(self.l_margin)

    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=(210 - w) / 2, w=w)
            self.ln(3)
        else:
            self.body(f"[Figur bulunamadi: {os.path.basename(path)}]")

# ASCII-guvenli yazim (fpdf2 Helvetica = Latin-1; Turkce ozel karakterler hata verir)
def _ascii(s):
    table = str.maketrans({
        "ş": "s", "Ş": "S",  # s,S (cedilla)
        "ğ": "g", "Ğ": "G",  # g,G (breve)
        "ı": "i", "İ": "I",  # i (dotless), I (dotted)
        "ö": "o", "Ö": "O",  # o,O (umlaut)
        "ü": "u", "Ü": "U",  # u,U (umlaut)
        "ç": "c", "Ç": "C",  # c,C (cedilla)
    })
    return str(s).translate(table).encode("latin-1", "replace").decode("latin-1")

pdf = KanserRefinementReport()
pdf.alias_nb_pages()
pdf.add_page()

# --- Baslik ---
pdf.set_font("Helvetica", "B", 18)
pdf.cell(0, 15, "KANSER Paneli: Refinement Raporu", 0, 1, "C")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 6, f"NB29 | {datetime.now().strftime('%Y-%m-%d')}", 0, 1, "C")
pdf.ln(5)

# --- Yonetici ozeti ---
pdf.section("Yonetici Ozeti")
best = results_df.iloc[0]
delta_nb16 = best["Boot-mean"] - 0.716
verdict = "NB16 referansi GECILDI" if delta_nb16 > 0 else "NB16 referansi gecilemedi"
pdf.body(_ascii(
    f"KANSER (hereditary cancer) panelinin %80/20 F1 skorunu yukseltmek icin 6 deney (E0-E5) "
    f"karsilastirildi. En iyi: {best['Deney']} (LOO-MCC(prior)={best['LOO-MCC (prior)']}, "
    f"Boot-mean %80/20 F1={best['Boot-mean']} +/- {best['Boot-std']}, AUPRC={best['AUPRC']}). "
    f"NB16 referans: stack_lr %80/20 F1=0.716. Fark: {delta_nb16:+.4f} -> {verdict}."))
pdf.body(_ascii(
    "Metodoloji notu: Tum KANSER-uzeri deneyler (E0,E2,E3,E4,E5) gercek 5-fold out-of-fold (OOF) "
    "tahmin ile degerlendirildi (self-prediction leakage onlendi). E1 (COMBINED) modeli "
    "MASTER+PAH+CFTR'de egitilip KANSER'de test edildi (cross-panel, leakage yok)."))

# --- Bolum 1: Sonuc tablosu ---
pdf.section("1. Deney Karsilastirmasi (LOO-MCC prior sirali)")
headers = ["Deney", "n_tr", "MCC(raw)", "MCC(pri)", "Boot-F1", "Boot-std", "AUPRC", "Prec", "Recall", "FP", "FN"]
cw = [34, 12, 16, 16, 17, 15, 14, 13, 14, 10, 10]
data = []
for _, row in results_df.iterrows():
    data.append([
        _ascii(row["Deney"])[:24], int(row["n_train"]),
        f"{row['LOO-MCC (raw)']:.3f}", f"{row['LOO-MCC (prior)']:.3f}",
        f"{row['Boot-mean']:.3f}", f"{row['Boot-std']:.3f}",
        f"{row['AUPRC']:.3f}",
        f"{row['Precision']:.3f}", f"{row['Recall']:.3f}",
        int(row["FP"]), int(row["FN"])
    ])
pdf.add_table(headers, data, cw)
pdf.ln(3)
pdf.body("Birincil metrik: Bootstrap %80/20 pathogenic-F1 (N=50, final benign-agirlikli dagilim). "
         "AUPRC literatur birincil metrigidir (Lee et al. 2023).")

# --- Figurler ---
pdf.section("2. LOO-CV MCC (raw vs prior-shift)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_loo_mcc.png"))

pdf.add_page()
pdf.section("3. Bootstrap %80/20 F1")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig2_boot_mean.png"))

pdf.section("4. AUPRC (Literatur Birincil Metrigi)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_auprc.png"))

pdf.add_page()
pdf.section("5. Confusion Matrix (En Iyi 3 Deney)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_confusion.png"))

# --- Prior-shift etkisi ---
pdf.section("6. Prior-shift Etkisi (Saerens 2002)")
ps_rows = []
for name in all_results:
    if all_results[name] is None:
        continue
    raw = all_results[name]["loo_raw"]["mcc"]
    pri = all_results[name]["loo_prior"]["mcc"]
    ps_rows.append([_ascii(name)[:30], f"{raw:.4f}", f"{pri:.4f}", f"{pri-raw:+.4f}"])
pdf.add_table(["Deney", "MCC(raw)", "MCC(prior)", "delta"], ps_rows, [70, 35, 35, 35])
pdf.ln(2)
pdf.body(_ascii(
    "Prior-shift, %69 patho egitim dagilimindan %20 patho test dagilimina post-hoc adapte eder. "
    "Calibrate-then-shift (Alexandari 2020): kalibrasyon ONCE, prior-shift SONRA -> "
    "kalibresiz EM'in zararli etkisi onlenir (NB21/NB22 PAH bulgusu)."))

# --- Overfit kontrolu ---
pdf.section("7. Overfit Kontrolu (train F1 vs OOF F1)")
of_rows = []
for name in all_results:
    if all_results[name] is None:
        continue
    tf = all_results[name]["train"]["train_f1"]
    lf = all_results[name]["loo_prior"]["f1"]
    gap = tf - lf
    status = "OK" if gap < 0.15 else "ORTA" if gap < 0.25 else "YUKSEK"
    of_rows.append([_ascii(name)[:30], f"{tf:.4f}", f"{lf:.4f}", f"{gap:.4f}", status])
pdf.add_table(["Deney", "Train-F1", "OOF-F1", "Gap", "Durum"], of_rows, [62, 32, 32, 28, 26])

# --- Tartisma ---
pdf.add_page()
pdf.section("8. Tartisma ve Literatur Temeli")
pdf.body(_ascii(
    "KANSER paneli (n=388, pos=268/%69, neg=120) en dengeli alt-settir; n=120 benign CFTR'nin 5x "
    "kati -> bootstrap CI'lar dar ve istatistiksel olarak guvenilir (NB16 std=0.033)."))
pdf.bullet(_ascii("COMBINED pooling (E1): BioData Mining 2026 -- gene-specific veri kit oldugunda disease/COMBINED pooling kazanir (n=388 bu kategoride)."))
pdf.bullet(_ascii("BalancedBagging (E2): Chatterji 2022 NeurIPS minimax-optimal; overfit gap'i azaltir."))
pdf.bullet(_ascii("Calibrate-then-shift (E3): Alexandari 2020 ICML -- kalibrasyon sonrasi prior-shift."))
pdf.bullet(_ascii("Heterogeneous stack (E4): CTpredX 2025 + Lee 2023 -- in-silico skorlar ham feature, RF base, LR meta (GBM meta overfit eder)."))
pdf.bullet(_ascii("Missing-dual (E5): KANSER'e ozgu -- missing label-sinyali farki %35.6 (PAH'ta %0.4); AL_16..AL_25 leakage riski icin flag'li/flagsiz cift model."))
pdf.ln(2)
pdf.body(_ascii(
    f"NB16 referans: KANSER stack_lr %80/20 F1=0.716, CI=[0.67-0.77]. "
    f"En iyi NB29 deneyi: {best['Deney']} (Boot={best['Boot-mean']:.3f}). {verdict}."))
pdf.body("NB16 AUPRC olculmedi; AUPRC mutlak deger olarak raporlanir (uydurma baseline kullanilmaz).")

# --- Kaydet ---
pdf_path = os.path.join(REPORTS_DIR_NB, "NB29_kanser_refinement_report.pdf")
pdf.output(pdf_path)
print(f"PDF raporu olusturuldu: {pdf_path}")


PDF raporu olusturuldu: /Users/tefe/teknofest_model/teknofest_model/reports/NB29_kanser_refinement_report.pdf
